
# Inventory Simulation & Stress Testing

This notebook evaluates the inventory policy using the **final 60-day
chronological holdout** from the demand-forecasting pipeline.

## Evaluation integrity

- Demand forecasting is evaluated as **rolling one-step-ahead forecasting**.
- The final 60-day period (`2025-05-02` to `2025-06-30`) is held out from model
  selection.
- Final-test observations are **not used to calibrate inventory uncertainty**.
- Inventory uncertainty is estimated from development-period **out-of-fold (OOF)
  forecast errors**.
- The final holdout is used only for inventory-policy evaluation.
- The inventory simulation is a **frozen-policy simulation**: the forecast plan
  is fixed during the simulation rather than dynamically reforecasted every day.

This separation prevents test-period information from being reused during
inventory calibration.



## Inventory policy

The simulation uses periodic review:

- Lead time: **3 days**
- Review period: **7 days**
- Protection period: **10 days**
- Service-level z value: **1.65**

Because replenishment decisions are made every 7 days, the protection period
is modeled as:

`lead time + review period = 3 + 7 = 10 days`

Safety stock is calculated as:

`safety stock = z × OOF error standard deviation × sqrt(protection period)`

This calculation assumes daily forecast errors are approximately independent.

The reported service metric is **cycle service level**: the percentage of
simulated days on which demand can be fully served without a stockout.

Lost units represent demand that could not be fulfilled during stockout days.


In [ ]:

import sys
from pathlib import Path

import pandas as pd


# Locate the repository root instead of assuming the notebook's
# execution working directory is the repository root.
current_path = Path.cwd().resolve()

candidate_roots = [
    current_path,
    *current_path.parents,
]

repo_root = None

for candidate in candidate_roots:
    if (
        (candidate / "src" / "inventory_simulation.py").exists()
        and (candidate / "reports").exists()
    ):
        repo_root = candidate
        break

if repo_root is None:
    raise FileNotFoundError(
        "Could not locate repository root containing "
        "'src/inventory_simulation.py' and 'reports/'."
    )

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))


from src.inventory_simulation import (
    LEAD_TIME_DAYS,
    REVIEW_PERIOD_DAYS,
    SERVICE_LEVEL_Z,
    simulate_inventory,
)


OOF_PATH = (
    repo_root
    / "reports"
    / "gradient_boosting_oof_predictions.csv"
)

TEST_PATH = (
    repo_root
    / "reports"
    / "final_test_predictions.csv"
)

SCENARIO_PATH = (
    repo_root
    / "reports"
    / "inventory_scenario_summary.csv"
)

PRODUCT_PATH = (
    repo_root
    / "reports"
    / "inventory_scenario_by_product.csv"
)

print("Inventory simulation notebook loaded.")
print(f"Repository root: {repo_root}")
print(f"OOF artifact:     {OOF_PATH}")
print(f"Final-test:       {TEST_PATH}")


In [ ]:

oof = pd.read_csv(
    OOF_PATH,
    parse_dates=["date"],
)

test = pd.read_csv(
    TEST_PATH,
    parse_dates=["date"],
)

required_oof = {
    "product_id",
    "date",
    "error",
}

required_test = {
    "product_id",
    "date",
    "quantity",
    "prediction",
}

missing_oof = required_oof - set(oof.columns)
missing_test = required_test - set(test.columns)

if missing_oof:
    raise ValueError(
        f"OOF artifact missing columns: {sorted(missing_oof)}"
    )

if missing_test:
    raise ValueError(
        f"Final-test artifact missing columns: {sorted(missing_test)}"
    )

print(f"OOF rows:        {len(oof):,}")
print(f"Final-test rows: {len(test):,}")

print(
    f"Final-test range: "
    f"{test['date'].min().date()} "
    f"to "
    f"{test['date'].max().date()}"
)

print(
    f"Products: "
    f"{test['product_id'].nunique()}"
)


In [ ]:

error_stats = (
    oof.groupby("product_id")["error"]
    .agg(
        error_std="std",
        oof_rows="size",
    )
    .reset_index()
)

if error_stats["error_std"].isna().any():
    raise ValueError(
        "At least one product has missing OOF error "
        "standard deviation."
    )

print("OOF uncertainty calibration:")

display(
    error_stats.head()
)

print()

print(
    "Products with OOF uncertainty estimates: "
    f"{len(error_stats)}"
)

print(
    "Mean OOF error standard deviation: "
    f"{error_stats['error_std'].mean():.2f}"
)


In [ ]:

print("POLICY ASSUMPTIONS")
print("-" * 50)

print(
    f"Lead time:          "
    f"{LEAD_TIME_DAYS} days"
)

print(
    f"Review period:      "
    f"{REVIEW_PERIOD_DAYS} days"
)

print(
    f"Protection period:  "
    f"{LEAD_TIME_DAYS + REVIEW_PERIOD_DAYS} days"
)

print(
    f"Service-level z:    "
    f"{SERVICE_LEVEL_Z}"
)

print()

print(
    "Safety stock uses OOF forecast-error variability, "
    "not final-test residuals."
)



## Normal-demand simulation

For each product, the final-test forecast is treated as the frozen demand
plan. The OOF error standard deviation for that product is used to estimate
safety stock.

No final-test residual is used to estimate uncertainty.


In [ ]:

test_sim = test.rename(
    columns={
        "quantity": "actual_demand",
        "prediction": "forecast",
    }
)

normal_results = []

for product_id, group in test_sim.groupby("product_id"):

    stats = error_stats[
        error_stats["product_id"] == product_id
    ]

    if len(stats) != 1:
        raise ValueError(
            "Expected exactly one OOF error estimate "
            f"for {product_id}."
        )

    error_std = float(
        stats["error_std"].iloc[0]
    )

    result = simulate_inventory(
        group,
        error_std=error_std,
        demand_multiplier=1.0,
    )

    result["product_id"] = product_id

    result["oof_error_std"] = error_std

    result["oof_rows"] = int(
        stats["oof_rows"].iloc[0]
    )

    normal_results.append(result)

normal_results = (
    pd.DataFrame(normal_results)
    .sort_values("product_id")
    .reset_index(drop=True)
)

normal_results.head()


In [ ]:

normal_summary = pd.DataFrame(
    [
        {
            "scenario": "A - Normal demand",

            "products": int(
                normal_results["product_id"].nunique()
            ),

            "test_days": 60,

            "lead_time_days": LEAD_TIME_DAYS,

            "review_period_days": REVIEW_PERIOD_DAYS,

            "protection_period_days": (
                LEAD_TIME_DAYS
                + REVIEW_PERIOD_DAYS
            ),

            "service_level_z": SERVICE_LEVEL_Z,

            "mean_product_cycle_service_level_%": round(
                normal_results[
                    "cycle_service_level_%"
                ].mean(),
                2,
            ),

            "products_below_90pct_service": int(
                (
                    normal_results[
                        "cycle_service_level_%"
                    ]
                    < 90
                ).sum()
            ),

            "products_below_95pct_service": int(
                (
                    normal_results[
                        "cycle_service_level_%"
                    ]
                    < 95
                ).sum()
            ),

            "total_lost_units": round(
                normal_results[
                    "lost_units"
                ].sum(),
                1,
            ),

            "total_units_ordered": round(
                normal_results[
                    "total_units_ordered"
                ].sum(),
                1,
            ),

            "total_average_inventory": round(
                normal_results[
                    "avg_inventory"
                ].sum(),
                1,
            ),

            "mean_safety_stock": round(
                normal_results[
                    "safety_stock"
                ].mean(),
                1,
            ),

            "mean_oof_error_std": round(
                normal_results[
                    "oof_error_std"
                ].mean(),
                2,
            ),
        }
    ]
)

display(
    normal_summary.T
)



## Stress scenarios

The scenario artifact contains five policy tests:

| Scenario | Definition |
|---|---|
| A | Normal demand |
| B | Actual demand increased by 10% |
| C | Actual demand increased by 20% |
| D | Unexpected 3× demand spike for five days |
| E | Maximum inventory-position capacity set to 70% of the unconstrained target |

Scenario D changes actual demand only. It represents an unexpected demand
shock rather than a known promotion.

Scenario E is a simulation constraint on maximum inventory position. It is
not a claim about a literal physical warehouse capacity.


In [ ]:

scenario_summary = pd.read_csv(
    SCENARIO_PATH
)

scenario_summary


In [ ]:

display(
    scenario_summary[
        [
            "scenario",
            "mean_product_cycle_service_level_%",
            "products_below_90pct_service",
            "products_below_95pct_service",
            "total_lost_units",
            "total_units_ordered",
            "total_average_inventory",
        ]
    ]
)



## Scenario interpretation

The stress tests show how the frozen inventory policy behaves when actual
demand differs from the baseline forecast assumptions or when inventory
capacity is constrained.

The results should be interpreted as **policy stress-test evidence**, not as
guarantees of future service performance.

In particular:

- Higher sustained demand increases stockout exposure.
- A short, unexpected demand spike can create concentrated lost demand.
- Restricting inventory position to 70% of the unconstrained target materially
  changes service outcomes.
- The normal-demand result is evaluated on the final holdout, while uncertainty
  was calibrated exclusively from development OOF errors.


In [ ]:

figure_path = (
    repo_root
    / "reports"
    / "figures"
    / "12_inventory_scenarios.png"
)

if figure_path.exists():

    from IPython.display import (
        Image,
        display,
    )

    display(
        Image(
            filename=str(
                figure_path
            )
        )
    )

else:
    print(
        f"Figure not found: {figure_path}"
    )



## Limitations

This simulation is intentionally a controlled inventory-policy experiment.

Important limitations include:

1. The demand data are synthetic.
2. The forecast is treated as a frozen plan during the 60-day simulation;
   dynamic daily reforecasting is not modeled.
3. Lead time is fixed at 3 days and review period at 7 days.
4. Safety-stock scaling assumes approximately independent daily forecast errors.
5. No procurement cost, holding cost, stockout penalty, minimum order quantity,
   case-pack constraint, or supplier capacity is modeled.
6. Products are simulated independently; substitution and cross-product effects
   are not modeled.
7. Promotion information is treated as known at prediction time.
8. Cycle service level is reported; this is different from fill rate.
9. Scenario stress tests are not probabilistic forecasts of future demand.

These limitations define the scope of the conclusions that can be drawn from
the experiment.



## Reproducibility

The notebook is a reporting and analysis layer over the canonical scripts.

Recommended execution order:

```text
python src/run_time_series_cv.py
python src/run_final_test.py
python src/run_inventory_simulation.py
python src/run_inventory_scenarios.py
```

The canonical inventory logic lives in:

`src/inventory_simulation.py`

The notebook reads the generated artifacts rather than maintaining a separate
implementation of the inventory engine.
